# Qwen2.5-Omni Fine-Tuning Pipeline

Complete pipeline for fine-tuning Qwen2.5-Omni-7B for tool calling with the Strands SDK.

## Pipeline Overview

| Stage | Purpose | Output |
|-------|---------|--------|
| Data Generation | LLM-generated tool calling examples | `train.jsonl`, `test.jsonl` |
| Fine-Tuning | Train with Unsloth and LoRA | Adapter weights |
| Quantization | Convert to GGUF format | `model-q4_k_m.gguf` |
| Evaluation | Validate performance | Metrics report |

## Key Features

- **100% LLM Generation**: Claude 3.5 Sonnet v2 for maximum realism
- **Complete Tool Coverage**: All 15 tools from TechCar Model X agent
- **Optimized Training**: Unsloth for 2x faster training, 60% less memory

## Requirements

- GPU: 14GB+ VRAM (RTX 3090, 4070 Ti, A100)
- RAM: 32GB+ system memory  
- Storage: 100GB free space
- CUDA: 11.8 or 12.1
- AWS Bedrock access for data generation

## Environment Setup and Authentication

**IMPORTANT:** Set your AWS Bedrock bearer token in the cell below before proceeding.

In [ ]:
# AWS Configuration
import os

# REPLACE WITH YOUR ACTUAL BEARER TOKEN
# Get your token from AWS console or your administrator
os.environ['AWS_BEARER_TOKEN_BEDROCK'] = 'YOUR_BEARER_TOKEN_HERE'
os.environ['AWS_REGION'] = 'us-east-1'

# Configure data generation model (you can change this)
MODEL_CONFIG = {
    'model_id': 'us.anthropic.claude-3-5-sonnet-20241022-v2:0',  # Claude 3.5 Sonnet v2
    'max_tokens': 1000,
    'temperature': 0.7
}

# Install dependencies and verify setup
!pip install -q torch transformers>=4.52.3 accelerate
!pip install -q datasets trl peft boto3 httpx matplotlib
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" || pip install -q unsloth

In [ ]:
import sys
import os
import json
import torch
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from typing import Dict, List, Any
from collections import Counter

# Add utils to path
sys.path.append('./utils')

# Import utilities
from data_generator import DataGenerator, ToolRegistry
from trainer import ModelTrainer, TrainingConfig  
from quantizer import ModelQuantizer, QuantizationConfig
from evaluator import ModelEvaluator, EvaluationMetrics

# System information
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {torch.cuda.get_device_name()} ({vram_gb:.1f} GB VRAM)")

# Create directories
for directory in ["./data", "./models", "./outputs"]:
    Path(directory).mkdir(exist_ok=True)

# Update data generator to use configured model
if 'MODEL_CONFIG' in globals():
    # Update the default model ID in data generator
    import data_generator
    data_generator.DEFAULT_MODEL_ID = MODEL_CONFIG['model_id']
    data_generator.DEFAULT_MAX_TOKENS = MODEL_CONFIG['max_tokens'] 
    data_generator.DEFAULT_TEMPERATURE = MODEL_CONFIG['temperature']
    print(f"Data generator configured with: {MODEL_CONFIG['model_id']}")

In [ ]:
# Initialize data generator with Bedrock integration
registry = ToolRegistry()
generator = DataGenerator(registry)

# Verify configuration
has_llm_client = hasattr(generator, 'llm_client') and generator.llm_client is not None
print(f"LLM Client configured: {'✅' if has_llm_client else '❌'}")
print(f"Tools loaded: {len(registry.tools)}")

# Tool categories summary
tool_categories = {
    "Cockpit Controls": ["climate_control", "window_control", "seat_control", "lighting_control", "drive_mode"],
    "Information Services": ["vehicle_assistant", "calendar_assistant", "search_assistant"],
    "Multimodal Tools": ["analyze_image", "voice_input"],
    "Utility Tools": ["select_model"],
    "Calendar Sub-tools": ["create_appointment", "list_appointments", "get_agenda", "update_appointment"]
}

for category, tools in tool_categories.items():
    available = sum(1 for tool in tools if registry.get_tool(tool))
    print(f"{category}: {available}/{len(tools)} tools")

## Stage 1: Data Generation

Generate synthetic training data using Claude 3.5 Sonnet v2 for maximum realism.

In [ ]:
# Generate training and test datasets
train_size = 40
test_size = 20

generator.generate_dataset(
    num_examples=train_size,
    output_path="./data/train.jsonl"
)

generator.generate_dataset(
    num_examples=test_size,
    output_path="./data/test.jsonl"
)

print(f"Generated {train_size} training examples and {test_size} test examples")

In [ ]:
# Exploratory Data Analysis of Generated Training Data
import json
import matplotlib.pyplot as plt
from collections import Counter

# Load training data
with open("./data/train.jsonl", "r") as f:
    train_data = [json.loads(line) for line in f]

with open("./data/test.jsonl", "r") as f:
    test_data = [json.loads(line) for line in f]

print(f"Training examples: {len(train_data)}")
print(f"Test examples: {len(test_data)}")

# Analyze tool usage
tool_counts = Counter()
message_lengths = []
multimodal_count = 0

for example in train_data:
    for tool in example.get("tools", []):
        tool_counts[tool["name"]] += 1
    
    for msg in example.get("messages", []):
        if isinstance(msg.get("content"), str):
            message_lengths.append(len(msg["content"]))
        elif isinstance(msg.get("content"), list):
            multimodal_count += 1

print(f"Multimodal examples: {multimodal_count}")
print(f"Average message length: {sum(message_lengths)/len(message_lengths):.1f} characters")

# Visualize distributions
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
tools, counts = zip(*tool_counts.most_common())
plt.bar(range(len(tools)), counts)
plt.xticks(range(len(tools)), tools, rotation=45, ha='right')
plt.title('Tool Usage Distribution')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
plt.hist(message_lengths, bins=30, alpha=0.7)
plt.title('Message Length Distribution')
plt.xlabel('Characters')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

# Sample conversations
for i, example in enumerate(train_data[:2]):
    print(f"\nExample {i+1} - Tools: {[tool['name'] for tool in example.get('tools', [])]}")
    
    for msg in example.get('messages', []):
        role = msg.get('role', 'unknown')
        content = msg.get('content', '')
        
        if isinstance(content, str):
            print(f"{role}: {content[:100]}{'...' if len(content) > 100 else ''}")
        elif isinstance(content, list):
            for item in content:
                if isinstance(item, dict) and 'toolUse' in item:
                    tool_name = item['toolUse']['name']
                    tool_input = item['toolUse']['input']
                    print(f"{role}: Tool call - {tool_name}({tool_input})")

## Stage 2: Fine-Tuning with Unsloth

Fine-tune the model using LoRA for efficient training.

In [ ]:
# Configure training
config = TrainingConfig(
    model_name="Qwen/Qwen2.5-7B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    num_epochs=1,
    batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    output_dir="./models/fine-tuned",
    use_flash_attention=True,
    gradient_checkpointing=True
)

config.save("./models/training_config.json")

In [ ]:
# Initialize trainer and setup model
trainer = ModelTrainer(config)
trainer.setup_model()

In [ ]:
# Prepare datasets
train_dataset, eval_dataset = trainer.prepare_dataset("./data/train.jsonl")

In [ ]:
# Start training
training_history = trainer.train(train_dataset, eval_dataset)

In [ ]:
# Plot training metrics
import matplotlib.pyplot as plt

# Extract metrics from history
train_loss = [h['loss'] for h in training_history if 'loss' in h]
eval_loss = [h['eval_loss'] for h in training_history if 'eval_loss' in h]

if train_loss:
    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(train_loss)
    plt.title('Training Loss')
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    
    if eval_loss:
        plt.subplot(1, 2, 2)
        plt.plot(eval_loss)
        plt.title('Evaluation Loss')
        plt.xlabel('Evaluation Steps')
        plt.ylabel('Loss')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Merge LoRA weights with base model
merged_model = trainer.merge_and_save("./models/merged")

## Stage 3: Quantization to GGUF

Convert the model to GGUF format for deployment with llama.cpp.

In [ ]:
# Configure quantization
quant_config = QuantizationConfig(
    quantization_method="q4_k_m",
    use_mmap=True,
    include_mmproj=True
)

quantizer = ModelQuantizer(quant_config)

if not quantizer.check_dependencies():
    print("Install llama.cpp: git clone https://github.com/ggerganov/llama.cpp && cd llama.cpp && make")

In [ ]:
# Run quantization pipeline
results = quantizer.full_pipeline(
    model_path="./models/merged",
    output_dir="./outputs/gguf",
    model_name="qwen2.5-omni-finetuned"
)

# Display results
for key, path in results.items():
    if Path(path).exists():
        size_gb = Path(path).stat().st_size / (1024**3)
        print(f"{key}: {Path(path).name} ({size_gb:.2f} GB)")

## Stage 4: Evaluation

Evaluate the fine-tuned model's performance.

In [ ]:
# Start llama-server for evaluation
print(f"llama-server -m {results.get('quantized', 'model.gguf')} --host 0.0.0.0 --port 8080 -c 2048 -ngl 35")

In [ ]:
# Run evaluation
if evaluator.client:
    metrics = evaluator.run_full_evaluation()
    metrics.save("./outputs/evaluation_metrics.json")

## Deployment Commands

### Start llama.cpp server
```bash
llama-server -m model-q4_k_m.gguf --host 0.0.0.0 --port 8080 -c 2048 -ngl 35
```

### Python integration
```python
from strands.models.llamacpp import LlamaCppModel

model = LlamaCppModel(
    base_url="http://localhost:8080",
    params={'temperature': 0.7, 'max_tokens': 1024}
)
```

### Edge deployment
```bash
scp model-q4_k_m.gguf edge-device:/opt/models/
```

## Summary

Pipeline completed successfully. The model is now:
- Fine-tuned for tool calling with Strands SDK format
- Quantized to 4-bit for efficient edge deployment
- Ready for integration with the edge agent system

### Key Outputs

| File | Description | Size |
|------|-------------|------|
| `train.jsonl` | Training dataset | ~50MB |
| `fine-tuned/` | LoRA adapter weights | ~200MB |
| `model-q4_k_m.gguf` | Quantized model | ~4.7GB |
| `evaluation_metrics.json` | Performance metrics | <1MB |

### Next Steps

1. Deploy the quantized model to edge devices
2. Integrate with the Strands agent framework
3. Test in production environment
4. Monitor performance and iterate as needed